# Practical 5: Box-Jenkins ARIMA Modeling
## Dataset: catfish.xls
## Objective: Perform stationarity testing, determine ARIMA orders (p,d,q) from ACF/PACF plots, fit and forecast using ARIMA model

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
df = pd.read_csv("catfish.xls")

In [ ]:
df.head(5)

In [ ]:
df.shape

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date")

In [ ]:
df.head(5)

In [ ]:
sns.lineplot(df)
plt.ylabel("Catfish Sales")

### 📊 How to Read This Graph:
- Monthly catfish sales showing trend and some volatility.
- We need to check if this data is stationary before applying ARIMA.

In [ ]:
#ADF and KPSS Test on Raw Data
print("ADF Statistic:", adfuller(df["Total"])[0])
print("ADF p-value:", adfuller(df["Total"])[1])
print("KPSS p-value:", kpss(df["Total"])[1])

In [ ]:
#First Differencing for Stationarity (d=1)
diff_catfish = df["Total"].diff().dropna()

print("Differenced ADF p-value:", adfuller(diff_catfish)[1])
print("Differenced KPSS p-value:", kpss(diff_catfish)[1])

In [ ]:
#ACF Plot - used to determine MA order (q)
plt.figure(figsize = (20,5))
plt.grid()
plot_acf(diff_catfish, ax = plt.gca(), lags = 30)
plt.show()

In [ ]:
#PACF Plot - used to determine AR order (p)
plt.figure(figsize = (20,5))
plt.grid()
plot_pacf(diff_catfish, ax = plt.gca(), lags = 30)
plt.show()

### 📊 How to Read ACF & PACF for ARIMA Order Selection:
- **PACF**: Look at where bars first cut into the blue shaded region. That lag = AR order **p**.
- **ACF**: Look at where bars first cut into the blue shaded region. That lag = MA order **q**.
- **d** = number of times we differenced the data (here d=1).

In [ ]:
#Train Test Split (80% Train, 20% Test)
train_df = df[:int(df.shape[0]*0.8)]
test_df = df[int(df.shape[0]*0.8):]

In [ ]:
#Fit ARIMA(1, 1, 1) Model
from statsmodels.tsa.arima.model import ARIMA

model_arima = ARIMA(train_df["Total"], order=(1, 1, 1))
model_arima_fit = model_arima.fit()
print(model_arima_fit.summary())

### 📝 How to Read ARIMA Summary Table:
- **coef**: The learned AR and MA coefficient values.
- **P>|z|**: The p-value for each coefficient. If p < 0.05, that coefficient is statistically significant.
- **AIC**: Akaike Information Criterion. Lower AIC = better model.

In [ ]:
#Forecast and MAPE Evaluation
from sklearn.metrics import mean_absolute_percentage_error

forecast_arima = model_arima_fit.forecast(len(test_df))
mape = mean_absolute_percentage_error(test_df["Total"], forecast_arima)
print("ARIMA Forecast MAPE:", mape)

In [ ]:
#Plot Actual vs Forecast
plt.plot(df["Total"], label="Original Data")
plt.plot(forecast_arima, label="ARIMA Forecast")
plt.legend()
plt.title("Catfish Sales ARIMA Forecast")
plt.show()

### 📊 How to Read the ARIMA Forecast Plot:
- **Blue line** = full original data (train + test).
- **Orange line** = ARIMA model forecast for the test period.
- Compare how closely the orange line follows the blue line in the test region.